In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/neoai-2026-qualification-code-stars-homework/problems.parquet
/kaggle/input/competitions/neoai-2026-qualification-code-stars-homework/train_results.parquet
/kaggle/input/competitions/neoai-2026-qualification-code-stars-homework/contest.parquet
/kaggle/input/competitions/neoai-2026-qualification-code-stars-homework/train.parquet
/kaggle/input/competitions/neoai-2026-qualification-code-stars-homework/test.parquet
/kaggle/input/competitions/neoai-2026-qualification-code-stars-homework/baseline_submission.csv


In [2]:
train = pd.read_parquet('/kaggle/input/competitions/neoai-2026-qualification-code-stars-homework/train.parquet')
train_results = pd.read_parquet('/kaggle/input/competitions/neoai-2026-qualification-code-stars-homework/train_results.parquet')
problems = pd.read_parquet('/kaggle/input/competitions/neoai-2026-qualification-code-stars-homework/problems.parquet')
contest = pd.read_parquet('/kaggle/input/competitions/neoai-2026-qualification-code-stars-homework/contest.parquet')
test = pd.read_parquet('/kaggle/input/competitions/neoai-2026-qualification-code-stars-homework/test.parquet')
sample_sub =pd.read_csv("/kaggle/input/competitions/neoai-2026-qualification-code-stars-homework/baseline_submission.csv")

In [3]:
contest_features = contest.copy()

contest_features["contest_hour"] = (
    pd.to_datetime(contest_features["startTime"], unit="s")
    .dt.hour
)

contest_features["contest_dayofweek"] = (
    pd.to_datetime(contest_features["startTime"], unit="s")
    .dt.dayofweek
)

contest_features["contest_month"] = (
    pd.to_datetime(contest_features["startTime"], unit="s")
    .dt.month
)

In [4]:
problem_features = (
    problems.groupby("contestId")
    .agg(
        n_problems=("index", "count"),

        avg_problem_rating=("rating", "mean"),
        max_problem_rating=("rating", "max"),
        min_problem_rating=("rating", "min"),
        std_problem_rating=("rating", "std"),

        avg_problem_points=("points", "mean"),
        total_problem_points=("points", "sum"),
        max_problem_points=("points", "max"),
    )
    .reset_index()
)

problem_features = problem_features.fillna(0)

In [5]:
tag_dummies = problems["tags"].str.get_dummies(sep=",")

problem_tag_features = pd.concat(
    [problems[["contestId"]], tag_dummies],
    axis=1
)

problem_tag_features = (
    problem_tag_features
    .groupby("contestId")
    .sum()
    .reset_index()
)

In [6]:
contest_result_features = (
    train.groupby("contestId")
    .agg(
        contest_participants=("handle", "count"),

        contest_avg_rank=("rank", "mean"),
        contest_avg_points=("points", "mean"),

        contest_max_points=("points", "max"),
        contest_std_points=("points", "std"),

        contest_avg_penalty=("penalty", "mean"),
        contest_avg_hacks=("successfulHackCount", "mean")
    )
    .reset_index()
)

contest_result_features = contest_result_features.fillna(0)

In [7]:
user_features = (
    train.groupby("handle")
    .agg(
        contests_played=("contestId", "count"),

        avg_rank=("rank", "mean"),
        median_rank=("rank", "median"),
        best_rank=("rank", "min"),
        worst_rank=("rank", "max"),

        avg_points=("points", "mean"),
        max_points=("points", "max"),

        avg_penalty=("penalty", "mean"),

        total_success_hacks=("successfulHackCount", "sum"),
        total_unsuccess_hacks=("unsuccessfulHackCount", "sum"),

        avg_success_hacks=("successfulHackCount", "mean"),
        avg_unsuccess_hacks=("unsuccessfulHackCount", "mean"),
    )
    .reset_index()
)

user_features = user_features.fillna(0)

In [8]:
train["top10"] = (train["rank"] <= 10).astype(int)
train["top50"] = (train["rank"] <= 50).astype(int)
train["top100"] = (train["rank"] <= 100).astype(int)

placement_features = (
    train.groupby("handle")
    .agg(
        top10_count=("top10", "sum"),
        top50_count=("top50", "sum"),
        top100_count=("top100", "sum")
    )
    .reset_index()
)

placement_features["top10_rate"] = (
    placement_features["top10_count"]
)

placement_features["top50_rate"] = (
    placement_features["top50_count"]
)

placement_features["top100_rate"] = (
    placement_features["top100_count"]
)

In [9]:
division_features = (
    train.groupby(["handle", "division"])
    .agg(
        div_avg_rank=("rank", "mean"),
        div_avg_points=("points", "mean"),
        div_contests=("contestId", "count")
    )
    .reset_index()
)

In [10]:
problem_solving = (
    train_results.groupby(["contestId", "handle"])
    .agg(
        solved_problems=("problemIndex", "count"),

        avg_problem_score=("points", "mean"),
        total_problem_score=("points", "sum"),

        avg_rejected=("rejectedAttemptCount", "mean"),
        total_rejected=("rejectedAttemptCount", "sum"),

        avg_submit_time=("bestSubmissionTimeSeconds", "mean"),
        max_submit_time=("bestSubmissionTimeSeconds", "max")
    )
    .reset_index()
)

In [11]:
user_solving_features = (
    problem_solving.groupby("handle")
    .agg(
        avg_solved=("solved_problems", "mean"),
        max_solved=("solved_problems", "max"),

        avg_problem_score=("avg_problem_score", "mean"),
        avg_total_score=("total_problem_score", "mean"),

        avg_rejected=("avg_rejected", "mean"),

        avg_submit_time=("avg_submit_time", "mean")
    )
    .reset_index()
)

user_solving_features = user_solving_features.fillna(0)

In [12]:
contest_strength = (
    train.groupby("contestId")
    .agg(
        avg_user_rank=("rank", "mean"),
        avg_user_points=("points", "mean")
    )
    .reset_index()
)

In [13]:
train = train.sort_values(["handle", "contestId"])

recent_features = (
    train.groupby("handle")
    .tail(5)
    .groupby("handle")
    .agg(
        recent_avg_rank=("rank", "mean"),
        recent_avg_points=("points", "mean"),
        recent_avg_penalty=("penalty", "mean")
    )
    .reset_index()
)

In [14]:
def build_features(df):

    df = df.merge(
        contest_features,
        on=["contestId", "division"],
        how="left"
    )

    df = df.merge(
        problem_features,
        on="contestId",
        how="left"
    )

    df = df.merge(
        problem_tag_features,
        on="contestId",
        how="left"
    )

    df = df.merge(
        contest_result_features,
        on="contestId",
        how="left"
    )

    df = df.merge(
        user_features,
        on="handle",
        how="left"
    )

    df = df.merge(
        placement_features,
        on="handle",
        how="left"
    )

    df = df.merge(
        user_solving_features,
        on="handle",
        how="left"
    )

    df = df.merge(
        recent_features,
        on="handle",
        how="left"
    )

    df = df.merge(
        contest_strength,
        on="contestId",
        how="left"
    )

    df = df.merge(
        division_features,
        on=["handle", "division"],
        how="left"
    )

    return df

In [15]:
X_train = build_features(train)

X_test = (
    test
    .merge(
        contest[["contestId", "division"]],
        on="contestId",
        how="left"
    )
)

X_test = build_features(X_test)

In [16]:
y_train = X_train["rank"]

In [19]:
X_cols = []

for col in X_train.columns:
    if col in X_test.columns:
        X_cols.append(col)

X_train = X_train[X_cols]
X_test = X_test[X_cols]

In [26]:
X_train = X_train.drop(["contestId", "handle"], axis =1)
X_test= X_test.drop(["contestId", "handle"], axis = 1)

In [27]:
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns

num_cols = [
    col for col in X_train.select_dtypes(include=["number"]).columns
    if not set(X_train[col].dropna().unique()).issubset({0, 1})
]

In [33]:
from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder().fit(X_train[cat_cols])
X_train[cat_cols] = encoder.transform(X_train[cat_cols])
X_test[cat_cols] =encoder.transform(X_test[cat_cols])

In [36]:
X_train = X_train.drop("startTime", axis= 1)
X_test= X_test.drop("startTime", axis = 1)

In [37]:
from xgboost import XGBRegressor
xgb=  XGBRegressor(n_estimators =1000, learning_rate= 0.01,max_depth = 6, device= "gpu").fit(X_train, y_train)

In [ ]:
from sklearn.linear_model import LinearRegression
lin = LinearRegression().fit(X_train, y_train)

In [40]:
sample_sub["rank"] =lin.predict(X_test)
sample_sub["rank"] = sample_sub["rank"].astype(int)
sample_sub.to_csv("ml2.csv", index= False)

In [ ]:
# for df in [X_train, X_test]:

#     df["rank_per_contest"] = (
#         df["avg_rank"] /
#         (df["contests_played"] + 1)
#     )

#     df["points_per_contest"] = (
#         df["avg_points"] *
#         np.log1p(df["contests_played"])
#     )

#     df["solving_efficiency"] = (
#         df["avg_total_score"] /
#         (df["avg_rejected"] + 1)
#     )

#     df["hack_efficiency"] = (
#         df["total_success_hacks"] /
#         (df["total_unsuccess_hacks"] + 1)
#     )

#     df["contest_difficulty_gap"] = (
#         df["avg_points"] -
#         df["avg_problem_rating"]
#     )